# 01 — The Hugging Face ecosystem and pretrained tokenizers (Parts A and B)

Needs internet the first time (models are then cached). Your name(s): _..._

**How the exercises work.** Cells marked 🧩 contain `____` blanks: you complete them using the hints (they name the functions/attributes you need and where to find them in the docs). Each exercise is followed by a ✅ *check* cell that tells you immediately whether you got it right.

In [1]:
import sys, pathlib, json, time
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
%load_ext autoreload
%autoreload 2

## Part A — The libraries

Read the documentation pages listed in the handout, then answer **in your own words**:

1. Purpose of `tokenizers`, `transformers`, `datasets`, `huggingface_hub`?
2. `tokenizers` vs. `transformers`?
3. What is `AutoTokenizer`? 4. What is a pretrained tokenizer? 5. What does a tokenizer configuration contain?
6. What is the Hugging Face Hub? 7. Fast vs. slow tokenizer?

**Your answer:**

_(write here)_

In [2]:
import tokenizers, transformers, datasets, huggingface_hub
for lib in (tokenizers, transformers, datasets, huggingface_hub):
    print(f"{lib.__name__:18} {lib.__version__}")

tokenizers         0.22.2
transformers       5.15.0
datasets           5.0.1
huggingface_hub    1.28.0


## Part B — Load the three pretrained tokenizers

### 🧩 Exercise 1.1 — Load tokenizers with `AutoTokenizer`

Load GPT-2, BERT-base-uncased and FLAN-T5-small into the dictionary `tokenizers_` (model name → tokenizer).

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `AutoTokenizer.from_pretrained(<model name>)` downloads (first time) and builds the right tokenizer class.
* Docs: https://huggingface.co/docs/transformers/main_classes/tokenizer
* If the download fails, check your internet connection; the tokenizer is cached after the first success.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAMES = ["gpt2", "bert-base-uncased", "google/flan-t5-small"]

tokenizers_ = {}
for model_name in MODEL_NAMES:
    tokenizers_[model_name] = ____

print({name: type(tok).__name__ for name, tok in tokenizers_.items()})

In [ ]:
assert set(tokenizers_) == {"gpt2", "bert-base-uncased", "google/flan-t5-small"}
assert all(hasattr(t, "tokenize") for t in tokenizers_.values())
print("✅ Exercise 1.1 OK")

### 🧩 Exercise 1.2 — Interrogate a tokenizer

For every tokenizer print: vocabulary size, **total** size (including added tokens), the special-tokens map, and whether it is a fast tokenizer.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* Attributes: `vocab_size`, `special_tokens_map`, `is_fast`.
* The total number of entries (base vocabulary + added tokens) is `len(tokenizer)`.
* Compare the two sizes for each model: where do they differ?

In [ ]:
rows = []
for name, tok in tokenizers_.items():
    rows.append({
        "model": name,
        "class": type(tok).__name__,
        "vocab_size": tok.____,
        "len(tokenizer)": ____,
        "is_fast": tok.____,
        "special_tokens": tok.____,
    })
pd.DataFrame(rows)

In [ ]:
for name, tok in tokenizers_.items():
    assert tok.is_fast, f"{name} should be a fast tokenizer"
    assert len(tok) >= tok.vocab_size
    assert isinstance(tok.special_tokens_map, dict) and tok.special_tokens_map
print("✅ Exercise 1.2 OK")

### 🧩 Exercise 1.3 — Look inside a fast tokenizer

A fast tokenizer wraps a `tokenizers.Tokenizer`. Print the four components of each: **model**, **normalizer**, **pre-tokenizer**, **decoder**. (This is the pipeline diagram of the walkthrough.)

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* The wrapped object is `tok.backend_tokenizer`.
* Its components are attributes with the names above: `.model`, `.normalizer`, `.pre_tokenizer`, `.decoder`.
* A component can be `None` (no normalizer): print the class name only when it exists.

In [ ]:
for name, tok in tokenizers_.items():
    bt = tok.____
    parts = {"model": bt.____, "normalizer": bt.____, "pre_tokenizer": bt.____, "decoder": bt.____}
    print(name)
    for part, obj in parts.items():
        print(f"   {part:14} {type(obj).__name__ if obj is not None else None}")

In [ ]:
bt = tokenizers_["bert-base-uncased"].backend_tokenizer
assert type(bt.model).__name__ == "WordPiece" or type(bt.model).__name__ == "WordLevel" or bt.model is not None
assert bt.pre_tokenizer is not None and bt.decoder is not None
print("✅ Exercise 1.3 OK  -- now draw the 5-box pipeline for each model on paper")

### Configuration files on the Hub (needs internet)

Which files make up a tokenizer repository?

In [ ]:
from huggingface_hub import list_repo_files
for name in tokenizers_:
    try:
        files = [f for f in list_repo_files(name) if any(k in f for k in ("token", "vocab", "merges", "spiece", "config", "special"))]
    except Exception as e:
        files = [f"(could not list: {type(e).__name__})"]
    print(name, "->", files)

**Questions**
1. Why do different models use different tokenizers?
2. What makes a tokenizer *fast*?
3. Why do the vocabulary sizes differ? Why can `vocab_size` and `len(tokenizer)` differ?
4. What are special tokens, and why handle them separately from ordinary vocabulary entries?
5. Which configuration file would you need to share to reproduce a tokenizer exactly?

**Your answer:**

_(write here)_